# Jalora LoRA Training - Google Colab

This notebook trains a LoRA adapter for Jalora using your training data.

## Requirements
- GPU runtime (Tesla T4 or better)
- HuggingFace account (for Qwen models)
- Git repository access

In [ ]:
# 1. Set up environment
import os
import json
from pathlib import Path

# Install dependencies
import subprocess
subprocess.run(['pip', 'install', '-q', 'peft', 'transformers', 'accelerate', 'trl', 'datasets', 'sentencepiece'])
print('Dependencies installed')

In [ ]:
# 2. Mount Google Drive (to save training results)
from google.colab import drive
drive.mount('/content/drive')

# Set up working directory
WORK_DIR = '/content/jalora'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Working directory: {WORK_DIR}')

In [ ]:
# 3. Clone your repository
# Replace with your actual repo URL
REPO_URL = 'https://github.com/your-username/Jalora.git'

if not os.path.exists(f'{WORK_DIR}/.git'):
    subprocess.run(['git', 'clone', REPO_URL, WORK_DIR])
    print('Repository cloned')
else:
    subprocess.run(['git', 'pull'])
    print('Repository updated')

os.chdir(WORK_DIR)
print(f'Current directory: {os.getcwd()}')

In [ ]:
# 4. Get HuggingFace token
import getpass
HF_TOKEN = getpass.getpass('Enter HuggingFace token: ')
os.environ['HF_TOKEN'] = HF_TOKEN
print('HuggingFace token set')

In [ ]:
# 5. Load training data
import json

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

# Load training and validation data
TRAIN_PATH = 'training_data/jalora_train.jsonl'
VAL_PATH = 'training_data/jalora_val.jsonl'

train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)

print(f'Loaded {len(train_data)} training examples')
print(f'Loaded {len(val_data)} validation examples')

In [ ]:
# 6. Check GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 7. Training configuration
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model selection - choose based on GPU memory
MODEL_NAME = 'Qwen/Qwen1.5-7B-Instruct'  # 4-8GB VRAM needed
OUTPUT_DIR = '/content/drive/MyDrive/jalora_lora_adapters'

# Alternative models:
# 'Qwen/Qwen1.5-1.5B-Instruct'  - 2-4GB VRAM
# 'Qwen/Qwen1.5-0.5B-Instruct'    - 1-2GB VRAM
# 'mistralai/Mistral-7B-v0.1'     - 4-8GB VRAM

print(f'Model: {MODEL_NAME}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# 8. Load model and tokenizer
import torch

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    token=os.environ.get('HF_TOKEN')
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer loaded: {len(tokenizer)} tokens')

print('Loading model...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    device_map='auto',
    torch_dtype=torch.float16,
    token=os.environ.get('HF_TOKEN'),
    trust_remote_code=True
)
print('Model loaded')

In [ ]:
# 9. Configure LoRA
lora_config = LoraConfig(
    r=64,              # Rank
    lora_alpha=16,     # Alpha
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
print(f'LoRA configured: r={lora_config.r}, alpha={lora_config.lora_alpha}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# 10. Training function
from transformers import TrainingArguments, Trainer
from datasets import Dataset

def format_prompt(example):
    """Format training example as expected by the model."""
    instruction = example.get('instruction', '')
    input_text = example.get('input', '')
    output = example.get('output', '')
    
    if input_text:
        return f'<|im_start|>user\n{instruction}\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>'
    return f'<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>'

# Prepare dataset
formatted_train = [{'text': format_prompt(ex)} for ex in train_data]
train_dataset = Dataset.from_list(formatted_train)
print(f'Dataset prepared with {len(train_dataset)} samples')

In [ ]:
# 11. Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=100,
    eval_steps=50,
    evaluation_strategy='steps',
    save_total_limit=2,
    report_to='none',
    fp16=True,
    bf16=False,
    optim='adamw_torch',
    dataloader_num_workers=2,
)

print('Training arguments configured')

In [ ]:
# 12. Create trainer and start training
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
    tokenizer=tokenizer,
)

print('\nStarting training...')
print('Training progress will be saved to:', OUTPUT_DIR)
trainer.train()

In [ ]:
# 13. Save adapter
print('\nSaving LoRA adapter...')
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to {OUTPUT_DIR}')

In [ ]:
# 14. Test the trained model
from peft import PeftModel

# Load base model and merge adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16,
    token=os.environ.get('HF_TOKEN'),
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = model.merge_and_unload()

# Test prompt
test_prompt = 'You are Jalora, a helpful AI assistant. How are you today?'
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = merged_model.generate(**inputs, max_new_tokens=100)
print('Model response:', tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# 15. Export to GGUF (optional)
# Run this in a second terminal:
# !pip install llama-cpp-python
# !cd /content && git clone https://github.com/ggerganov/llama.cpp
# !cd llama.cpp && make -j8
# !python convert_hf_to_gguf.py <output_dir> --outfile jalora-qwen-7b.Q4_K_M.gguf
# !./llama.cpp/quantize jalora-qwen-7b.Q4_K_M.gguf jalora-qwen-7b.Q4_K_M.gguf q4_k_m

## Training Summary

### Training Progress
Monitor training in the output above.

### To Continue Training Later
1. Open this notebook again
2. Change epochs to continue: `num_train_epochs=6` (continue from epoch 3)
3. Or resume from checkpoint if saved

### Exporting for Ollama
1. Download `llama.cpp`
2. Convert model to GGUF
3. Quantize to Q4_K_M
4. Import to Ollama with Modelfile